# 🍜 Recipe → Molecule 분석 파이프라인 (재설계)

이 노트북은 **recipe–molecule strength / 확률 생성 단계**를 중심으로 파이프라인을 **새로 재설계**합니다.
기존 병목이었던 다음 문제를 전단에서 해결하는 것이 목표입니다.

- TOPK 고정으로 인한 recipe 표현 평탄화
- binary co-occurrence로 인한 범용 분자쌍 과강화
- FlavorDB 매핑 ingredient만으로 ratio를 재정규화하여 cuisine-specific 질량 소실

핵심 목표:
- cuisine별 top molecule overlap(Jaccard)을 **0.3–0.6** 수준으로 낮출 만큼, 전단에서 정보가 살아있게 만들기

---
## ✅ 새 정의(요약)

레시피 $r$, molecule $m$, ingredient $i$에 대해 다음을 사용합니다.

1) 레시피의 원래 ratio는 **절대 보존**:
$$\sum_{i\in J(r)} w_{r,i}=1,\quad w_{r,i}\ge 0$$

2) FlavorDB 매핑 실패 질량은 UNK로 보존:
$$u_r:=\sum_{i\in J(r)\setminus I(r)} w_{r,i}$$

3) 전역적으로 흔한 molecule을 downweight하는 가중치 $g(m)$ (IDF류):
$$g(m)=\left(\log\frac{N+1}{df(m)+1}\right)^{\beta}$$

4) ingredient→molecule 분배 커널:
$$q(m\mid i)=\frac{\mathbf{1}\{m\in M(i)\}\, g(m)}{\sum\limits_{m'\in M(i)} g(m')}$$

5) 레시피 raw score:
$$S_0(r,m)=\sum_{i\in I(r)} w_{r,i}\,q(m\mid i)$$

6) 레시피별 확률 (샤프닝 포함):
$$p(m\mid r)=(1-u_r)\cdot\frac{S_0(r,m)^{\tau}}{\sum_{m'} S_0(r,m')^{\tau}},\quad p(\mathrm{UNK}\mid r)=u_r$$

7) TOPK 고정 대신 **누적 질량 컷**(예: $\rho=0.9$):
$$\sum_{m\in\mathcal{K}_r} p(m\mid r)\ge \rho$$


## 0. 환경 설정
- 아래 셀에서 필요한 라이브러리를 로드합니다.
- 데이터 경로는 본인 환경에 맞게 수정하세요.


In [1]:
import os
import json
import math
import re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

# 시각화 (필요할 때만 사용)
import matplotlib.pyplot as plt

# 선택: 그래프/클러스터링
# !pip install networkx scikit-learn
import networkx as nx
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering


## 1. 데이터 로드

필요한 입력은 크게 3종입니다.

1) **레시피 데이터**: recipe_id, cuisine, ingredient, amount(grams) 또는 ratio
2) **FlavorDB 매핑 데이터**: ingredient(alias) → molecule set
3) (선택) tag/descriptor 데이터: molecule 또는 ingredient의 태그/설명

> ⚠️ 이 노트북은 *형식 의존도를 낮추기* 위해, 아래에서 공통 표준형으로 변환합니다.


In [2]:
# === 경로 설정 ===
DATA_DIR = Path("./data")   # 필요시 변경

# 예시 파일명 (본인 파일명에 맞게 수정)
RECIPE_CSV = DATA_DIR / "recipe_with_ratio.csv"   # columns 예: recipe_id, cuisine, ingredient, grams
FLAVOR_ALIAS_CSV = DATA_DIR / "flavordb_alias_in_vocab.csv"  # columns 예: alias, molecules (set-like string)

# 로드
recipes = pd.read_csv(RECIPE_CSV)
alias = pd.read_csv(FLAVOR_ALIAS_CSV)

print("recipes:", recipes.shape)
print("alias:", alias.shape)
recipes.head(3)


recipes: (554161, 4)
alias: (814, 7)


,name,cleaned_ingredients,ingredients_ratio,cuisine
0,Worlds Best Mac and Cheese,"[['penne', 170.0], ['cheddar cheese', 28.0], [...","[['cheddar cheese', 0.0208], ['gruyere cheese'...",Italian
1,Dilly Macaroni Salad Recipe,"[['macaroni', 140.0], ['cheese', 113.0], ['cel...","[['macaroni', 0.4029], ['cheese', 0.3252], ['c...",American
2,Gazpacho,"[['tomato', 1600.0], ['salt', 3.0], ['onion', ...","[['tomato', 0.7976], ['salt', 0.0015], ['onion...",French


## 2. 표준화: 레시피를 $w_{r,i}$ (ratio) 형태로 만들기

레시피 데이터가 grams를 가지고 있으면 recipe별 총량으로 나눠 ratio를 만듭니다.

- 목표: 각 recipe_id에 대해 $\sum_i w_{r,i}=1$
- grams가 이미 ratio면, 그대로 사용하세요.


In [3]:
import ast
import numpy as np
import pandas as pd

df_wide = recipes.copy()

# recipe_id 없으면 생성
if "recipe_id" not in df_wide.columns:
    df_wide = df_wide.reset_index().rename(columns={"index":"recipe_id"})

# 1) cleaned_ingredients를 파싱해서 "list of [ing, grams]"로 통일
def parse_list_of_pairs(x):
    if pd.isna(x):
        return []
    # 문자열이면 literal_eval
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception:
            return []
    # 이제 x가 list여야 함
    if not isinstance(x, list):
        return []
    # x 안에 pair만 남기기
    out = []
    for t in x:
        if isinstance(t, (list, tuple)) and len(t) == 2:
            ing = str(t[0])
            try:
                g = float(t[1])
            except Exception:
                continue
            out.append([ing, g])
    return out

df_wide["cleaned_ingredients_parsed"] = df_wide["cleaned_ingredients"].apply(parse_list_of_pairs)

# 2) explode 해서 각 row가 pair가 되게 만들기
df_long = (
    df_wide[["recipe_id","name","cuisine","cleaned_ingredients_parsed"]]
    .explode("cleaned_ingredients_parsed")
    .dropna(subset=["cleaned_ingredients_parsed"])
    .copy()
)

# 3) pair -> ingredient, grams
df_long[["ingredient","grams"]] = pd.DataFrame(
    df_long["cleaned_ingredients_parsed"].tolist(),
    index=df_long.index
)
df_long["grams"] = df_long["grams"].astype(float)

# (선택) 같은 ingredient가 여러 번 나오면 합치기
df_long = (
    df_long.groupby(["recipe_id","name","cuisine","ingredient"], as_index=False)
           .agg({"grams":"sum"})
)

# 4) 전체 grams 기준 ratio(w) 만들기 (FlavorDB 매핑과 무관)
total = df_long.groupby("recipe_id")["grams"].transform("sum")
df_long = df_long.loc[total > 0].copy()
df_long["w"] = df_long["grams"] / total

print(df_long.shape)
print(df_long.groupby("recipe_id")["w"].sum().describe())
df_long.head(3)

(4717247, 6)
count    5.530500e+05
mean     1.000000e+00
std      3.148764e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: w, dtype: float64


,recipe_id,name,cuisine,ingredient,grams,w
0,0,Worlds Best Mac and Cheese,Italian,butter,113.0,0.074538
1,0,Worlds Best Mac and Cheese,Italian,cheddar cheese,28.0,0.018470
2,0,Worlds Best Mac and Cheese,Italian,cheese,452.0,0.298153


In [4]:
total = df_long.groupby("recipe_id")["grams"].transform("sum")
df_long = df_long.loc[total > 0].copy()
df_long["w"] = df_long["grams"] / total

print(df_long.groupby("recipe_id")["w"].sum().describe())

count    5.530500e+05
mean     1.000000e+00
std      3.148764e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: w, dtype: float64


In [5]:
recipes = df_long.copy()

## 3. FlavorDB molecules 파싱: ingredient → molecule set $M(i)$

alias 데이터의 molecules 컬럼이 다음과 같은 형태일 수 있습니다.

- 예: `"{27457, 31252, 7976}"`

이를 파싱하여 `dict[str, set[int]]` 형태로 만듭니다.


In [6]:
def parse_molecule_set(x):
    """molecules 문자열을 set[int]로 파싱. 실패 시 빈 set 반환."""
    if pd.isna(x):
        return set()
    s = str(x).strip()
    # { ... } 형태 가정
    s = s.strip("{}").strip()
    if len(s) == 0:
        return set()
    # 숫자 추출
    nums = re.findall(r"\d+", s)
    return set(int(t) for t in nums)

# alias 표준 컬럼명 지정 (필요시 수정)
assert "alias" in alias.columns
assert "molecules" in alias.columns

alias["mol_set"] = alias["molecules"].apply(parse_molecule_set)
print(alias["mol_set"].apply(len).describe())

ing2mols = dict(zip(alias["alias"].astype(str), alias["mol_set"]))

# 예시 확인
sample_keys = list(ing2mols.keys())[:5]
for k in sample_keys:
    print(k, list(sorted(ing2mols[k]))[:10], "..., |M|=", len(ing2mols[k]))


count    814.00000
mean      66.59828
std       66.05741
min        1.00000
25%        4.00000
50%       51.50000
75%      106.75000
max      385.00000
Name: mol_set, dtype: float64
bread [247, 1130, 8094] ..., |M|= 3
rye bread [72, 332, 460, 999, 1049, 1130, 1183, 6202, 6561, 7344] ..., |M|= 30
white bread [880, 994, 5960, 6140, 7359, 7361, 7362, 10883, 11173, 12170] ..., |M|= 11
whole wheat flour [179, 247, 612, 878, 957, 1068, 1130, 3776, 6202, 6561] ..., |M|= 56
arrack [240, 1031, 6584, 7997, 31249] ..., |M|= 5


## 4. 매핑 여부에 따른 mass 보존: $u_r$ (UNK 질량)

- $J(r)$: 레시피에 등장한 전체 ingredient 집합
- $I(r)$: FlavorDB에 매핑되는 ingredient 집합

UNK 질량:
$u_r=\sum_{i\in J(r)\setminus I(r)} w_{r,i}$

이 값은 **나중에 확률에서 그대로 보존**합니다.


In [7]:
# 매핑 여부 플래그
recipes["is_mapped"] = recipes["ingredient"].astype(str).map(lambda x: x in ing2mols and len(ing2mols[x]) > 0)

# recipe별 UNK 질량
unk_mass = recipes.loc[~recipes["is_mapped"]].groupby("recipe_id")["w"].sum()
unk_mass = unk_mass.reindex(recipes["recipe_id"].unique(), fill_value=0.0)

print("UNK mass stats:")
print(unk_mass.describe())


UNK mass stats:
count    553050.000000
mean          0.389098
std           0.287065
min           0.000000
25%           0.140190
50%           0.345220
75%           0.604627
max           1.000000
Name: w, dtype: float64


## 5. 전역 흔함 억제: $df(m)$ 및 $g(m)$

여기서 $df(m)$는 “molecule $m$가 **등장 가능한** 레시피 수”로 정의합니다.

- 레시피 $r$에서 매핑된 ingredient들 중 하나라도 $m\in M(i)$이면 등장으로 카운트.

$$df(m)=\left|\{r: \exists i\in I(r),\; m\in M(i)\}\right|$$

그리고 IDF류:
$$g(m)=\left(\log\frac{N+1}{df(m)+1}\right)^{\beta}$$

- $\beta$: downweight 강도 (기본 1)


In [8]:
# 파라미터
beta = 1.0

# recipe별 mapped ingredient 목록
mapped = recipes.loc[recipes["is_mapped"], ["recipe_id", "ingredient"]].drop_duplicates()

# recipe -> molecule set union
recipe2molset = defaultdict(set)
for rid, ing in mapped.itertuples(index=False):
    recipe2molset[rid].update(ing2mols[str(ing)])

N = len(recipe2molset)  # mapped ingredient가 1개 이상 있는 레시피 수 기준
print("N (recipes with >=1 mapped ingredient):", N)

# df(m) 계산
df = Counter()
for rid, mols in recipe2molset.items():
    for m in mols:
        df[m] += 1

df_series = pd.Series(df)
print(df_series.describe())

def g_idf(m, beta=1.0):
    return (math.log((N + 1) / (df.get(m, 0) + 1))) ** beta

# g(m) 미리 계산 (속도)
all_mols = list(df.keys())
# g = {m: g_idf(m, beta=beta) for m in all_mols}
g = {m: 1 for m in all_mols}

# sanity: 상위/하위 df 확인
top_df = df_series.sort_values(ascending=False).head(10)
low_df = df_series.sort_values(ascending=True).head(10)
print("\nTop df molecules (very common):")
print(top_df)
print("\nLow df molecules (rare):")
print(low_df)


N (recipes with >=1 mapped ingredient): 546529
count      1374.000000
mean     109064.235808
std      140146.664421
min           1.000000
25%       11076.000000
50%       46233.000000
75%      150847.250000
max      539948.000000
dtype: float64

Top df molecules (very common):
644104    539948
8094      539660
1130      538002
6202      525587
247       524863
31260     519477
11552     518740
6054      518464
650       517289
798       515974
dtype: int64

Low df molecules (rare):
5280450     1
13854       1
13849       1
11815386    1
8299        1
7768        1
887         1
61093       1
16296       1
61092       1
dtype: int64


## 6. ingredient→molecule 분배 커널 $q(m\mid i)$

$$q(m\mid i)=\frac{\mathbf{1}\{m\in M(i)\}\, g(m)}{\sum_{m'\in M(i)} g(m')}$$

- ingredient 질량은 유지하되, 그 ingredient가 가진 molecule 중에서도 **전역적으로 덜 흔한 분자**에 더 mass를 할당합니다.

> 만약 어떤 ingredient에서 $\sum_{m'\in M(i)} g(m')=0$인 경우(극단적/예외), 균등 분배로 fallback 합니다.


In [9]:
def q_m_given_i(m, molset_i, g_dict):
    denom = sum(g_dict.get(mm, 0.0) for mm in molset_i)
    if denom <= 0:
        # fallback: uniform
        return 1.0 / max(1, len(molset_i)) if m in molset_i else 0.0
    return (g_dict.get(m, 0.0) / denom) if m in molset_i else 0.0


## 7. 레시피 raw score $S_0(r,m)$ 및 확률 $p(m\mid r)$

1) raw score:
$$S_0(r,m)=\sum_{i\in I(r)} w_{r,i}\,q(m\mid i)$$

2) 샤프닝 포함 확률:
$$p(m\mid r)=(1-u_r)\cdot\frac{S_0(r,m)^{\tau}}{\sum_{m'} S_0(r,m')^{\tau}},\quad p(\mathrm{UNK}\mid r)=u_r$$

- $\tau>1$: 분포를 더 뾰족하게(레시피 특이성 증가)
- $\tau<1$: 더 퍼지게


In [10]:
tau = 1.4 # 1.2  # 기본 샤프닝

# 레시피별 ingredient list (w 포함)
recipe_groups = recipes.groupby("recipe_id")

def compute_p_m_given_r(rid):
    grp = recipe_groups.get_group(rid)
    # UNK 질량
    u = float(grp.loc[~grp["is_mapped"], "w"].sum())

    # mapped ingredient만 사용
    mapped_rows = grp.loc[grp["is_mapped"], ["ingredient", "w"]]
    if len(mapped_rows) == 0:
        # 전부 UNK면 UNK=1
        return {"UNK": 1.0}

    # S0 누적
    S0 = defaultdict(float)
    for ing, w in mapped_rows.itertuples(index=False):
        molset = ing2mols[str(ing)]
        denom = sum(g.get(mm, 0.0) for mm in molset)
        if denom <= 0:
            # uniform fallback
            for m in molset:
                S0[m] += float(w) * (1.0 / max(1, len(molset)))
        else:
            for m in molset:
                S0[m] += float(w) * (g.get(m, 0.0) / denom)

    # 샤프닝 + 정규화 (UNK 제외 파트의 총질량을 (1-u)로 맞춤)
    if len(S0) == 0:
        return {"UNK": 1.0}

    # power
    S_pow = {m: (v ** tau) for m, v in S0.items() if v > 0}
    Z = sum(S_pow.values())
    if Z <= 0:
        return {"UNK": 1.0}

    p = {m: (1.0 - u) * (v / Z) for m, v in S_pow.items()}
    p["UNK"] = u

    # p = {m:(v / Z) for m, v in S_pow.items()}
    # p["UNK"] = 0
    return p

# 샘플 계산
sample_rids = recipes["recipe_id"].unique()[:3]
for rid in sample_rids:
    p = compute_p_m_given_r(rid)
    # 상위 10개
    top10 = sorted([(k,v) for k,v in p.items() if k!="UNK"], key=lambda x: x[1], reverse=True)[:10]
    print("rid:", rid, "UNK:", p.get("UNK", 0.0))
    print("top10:", top10)
    print()


rid: 0 UNK: 0.11345646437994723
top10: [(8094, 0.010138078578343342), (8194, 0.009312732875149029), (1032, 0.009312732875149029), (12813, 0.009312732875149029), (31260, 0.009312732875149029), (6184, 0.009312732875149029), (1068, 0.009312732875149029), (31276, 0.009312732875149029), (31289, 0.009312732875149029), (7797, 0.009312732875149029)]

rid: 1 UNK: 0.4095157179269329
top10: [(994, 0.15135860295254774), (6140, 0.15135860295254774), (6274, 0.15135860295254774), (8094, 0.002940635432664314), (644104, 0.001571182318749507), (6202, 0.001571182318749507), (1130, 0.001571182318749507), (31260, 0.0014301551251154456), (126, 0.0014301551251154456), (650, 0.0014301551251154456)]

rid: 2 UNK: 0.18322475570032573
top10: [(644104, 0.004769589047903027), (1130, 0.004769589047903027), (527, 0.0042933324883870005), (8723, 0.0042933324883870005), (31260, 0.0042933324883870005), (15394, 0.0042933324883870005), (6184, 0.0042933324883870005), (439341, 0.0042933324883870005), (5365811, 0.004293332488

## 8. TOPK 제거: 누적 질량 컷 $\rho$

레시피마다 분포가 다르므로, 고정 K가 아니라 누적 질량 기준으로 보관합니다.

$$\sum_{m\in\mathcal{K}_r} p(m\mid r)\ge \rho$$

- 권장: $\rho\in[0.85, 0.95]$


In [11]:
rho = 0.90

def mass_cut(p_dict, rho=0.9, include_unk=False):
    items = [(m, v) for m, v in p_dict.items() if (include_unk or m != "UNK")]
    items.sort(key=lambda x: x[1], reverse=True)
    kept = []
    s = 0.0
    for m, v in items:
        kept.append((m, v))
        s += v
        if s >= rho:
            break
    return kept, s

rid = recipes["recipe_id"].unique()[0]
p = compute_p_m_given_r(rid)
kept, s = mass_cut(p, rho=rho, include_unk=False)
print("kept size:", len(kept), "mass:", s, "UNK:", p.get("UNK",0.0))
kept[:10]


kept size: 254 mass: 0.8865435356200515 UNK: 0.11345646437994723


[(8094, 0.010138078578343342),
 (8194, 0.009312732875149029),
 (1032, 0.009312732875149029),
 (12813, 0.009312732875149029),
 (31260, 0.009312732875149029),
 (6184, 0.009312732875149029),
 (1068, 0.009312732875149029),
 (31276, 0.009312732875149029),
 (31289, 0.009312732875149029),
 (7797, 0.009312732875149029)]

## 9. 전단 품질 진단 지표

### 9.1 recipe-level entropy 및 effective support

- entropy:
$$H(r)=-\sum_m p(m\mid r)\log p(m\mid r)$$
- effective support:
$$\exp(H(r))$$

### 9.2 global top molecule mass

전역 상위 G개 molecule이 레시피에서 차지하는 평균 질량이 줄어야 합니다.

### 9.3 cuisine별 top molecule overlap (Jaccard)

cuisine별 집계 후 상위 set을 뽑아 Jaccard를 봅니다.


In [12]:
def entropy(p_dict, eps=1e-12):
    vals = np.array([v for k,v in p_dict.items() if k!="UNK" and v>0], dtype=float)
    if len(vals)==0:
        return 0.0
    return float(-(vals * np.log(vals + eps)).sum())

# 전체 레시피에 대해 p 계산 (시간이 걸리면 샘플로 시작)
unique_rids = recipes["recipe_id"].unique()

p_by_r = {}
for rid in unique_rids:
    p_by_r[rid] = compute_p_m_given_r(rid)

H = np.array([entropy(p_by_r[rid]) for rid in unique_rids])
eff = np.exp(H)
print("Entropy stats:", pd.Series(H).describe())
print("Effective support stats:", pd.Series(eff).describe())

# 전역 top molecule
mol_mass = Counter()
for rid, p in p_by_r.items():
    for m,v in p.items():
        if m=="UNK": 
            continue
        mol_mass[m] += v
mol_mass_series = pd.Series(mol_mass).sort_values(ascending=False)

G = 50
topG = set(mol_mass_series.head(G).index.tolist())
avg_topG_mass = np.mean([sum(v for m,v in p_by_r[rid].items() if m in topG) for rid in unique_rids])
print(f"Avg mass on global Top{G} molecules:", avg_topG_mass)


Entropy stats: count    5.530500e+05
mean     2.402546e+00
std      1.312945e+00
min     -1.000089e-12
25%      1.400902e+00
50%      2.202789e+00
75%      3.351943e+00
max      5.987707e+00
dtype: float64
Effective support stats: count    553050.000000
mean         26.791300
std          41.920513
min           1.000000
25%           4.058859
50%           9.050218
75%          28.558161
max         398.499752
dtype: float64
Avg mass on global Top50 molecules: 0.36679162586125946


In [13]:
# cuisine별 aggregation (mean)
rid2cuisine = recipes.groupby("recipe_id")["cuisine"].first().to_dict()

cuisine2mass = defaultdict(Counter)
cuisine2cnt = Counter()

for rid, p in p_by_r.items():
    c = rid2cuisine.get(rid, "unknown")
    cuisine2cnt[c] += 1
    for m,v in p.items():
        if m=="UNK": 
            continue
        cuisine2mass[c][m] += v

# mean으로 변환
cuisine2mean = {}
for c, mass in cuisine2mass.items():
    n = cuisine2cnt[c]
    cuisine2mean[c] = {m: v/n for m,v in mass.items()}

# cuisine별 top set 뽑기: 누적질량 기준
def top_set_from_dist(dist, rho=0.9):
    items = sorted(dist.items(), key=lambda x:x[1], reverse=True)
    s=0.0
    out=set()
    for m,v in items:
        out.add(m)
        s += v
        if s>=rho:
            break
    return out

cuisines = sorted(cuisine2mean.keys())
# top_sets = {c: top_set_from_dist(cuisine2mean[c], rho=0.9) for c in cuisines}
def topK_from_dist(dist, K=100):
    return set(
        m for m,_ in sorted(dist.items(), key=lambda x:x[1], reverse=True)[:K]
    )

top_sets = {
    c: topK_from_dist(cuisine2mean[c], K=100)
    for c in cuisines
}


def jaccard(a,b):
    if len(a|b)==0: return 0.0
    return len(a&b)/len(a|b)

# 상위 몇 개 cuisine pair만 출력
pairs = []
for i in range(len(cuisines)):
    for j in range(i+1, len(cuisines)):
        c1,c2 = cuisines[i], cuisines[j]
        pairs.append((jaccard(top_sets[c1], top_sets[c2]), c1, c2, len(top_sets[c1]), len(top_sets[c2])))
pairs.sort(reverse=True)
print("Top overlaps (should go DOWN after redesign):")
for score,c1,c2,k1,k2 in pairs[:10]:
    print(f"{c1} vs {c2}: J={score:.3f} (|K|={k1},{k2})")


Top overlaps (should go DOWN after redesign):
Korean vs Spanish: J=0.980 (|K|=100,100)
Indian vs Spanish: J=0.961 (|K|=100,100)
Indian vs Korean: J=0.961 (|K|=100,100)
Latin vs Spanish: J=0.942 (|K|=100,100)
Japanese vs Vietnamese: J=0.942 (|K|=100,100)
Chinese vs Korean: J=0.942 (|K|=100,100)
Thai vs Vietnamese: J=0.923 (|K|=100,100)
Malaysian vs Vietnamese: J=0.923 (|K|=100,100)
Korean vs Latin: J=0.923 (|K|=100,100)
Japanese vs Malaysian: J=0.923 (|K|=100,100)


In [14]:
import pickle
from pathlib import Path

SAVE_DIR = Path("./saved_state")
SAVE_DIR.mkdir(exist_ok=True)

with open(SAVE_DIR / "p_by_r.pkl", "wb") as f:
    pickle.dump(p_by_r, f)

with open(SAVE_DIR / "rid2cuisine.pkl", "wb") as f:
    pickle.dump(rid2cuisine, f)

# (선택) recipes 자체도 저장
recipes.to_pickle(SAVE_DIR / "recipes_long.pkl")

print("Saved!")


Saved!


In [15]:
import pickle
from pathlib import Path

SAVE_DIR = Path("./saved_state")

with open(SAVE_DIR / "p_by_r.pkl", "rb") as f:
    p_by_r = pickle.load(f)

with open(SAVE_DIR / "rid2cuisine.pkl", "rb") as f:
    rid2cuisine = pickle.load(f)

recipes = pd.read_pickle(SAVE_DIR / "recipes_long.pkl")

print(len(p_by_r), len(rid2cuisine))


553050 553050
